# Model Compression Experiments
### Pruning · Quantization · Low-Rank Factorization across 9 Models

| Task | CNN | Transformer | RNN | Dataset |
|---|---|---|---|---|
| Audio Classification | M3 | AST | AudioLSTM | Speech Commands |
| Sentiment Analysis | TextCNN | DistilBERT | BiLSTM | IMDB |
| Machine Translation | ConvS2S | TranslationTransformer | GRUSeq2Seq | IWSLT2017 de-en |

**Compression rates:** 15%, 30%, 45%, 60%, 75%  
**Train / Test split:** 70 / 30

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torchtext datasets sacrebleu pandas numpy

In [ ]:
!pip install --upgrade pillow transformers

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

# =============================================================================
# Compression Helper Methods
# =============================================================================

def _masked_stats(model):
    """
    Returns {param_id: (numel, nonzero)} for every parameter that currently
    has an active pruning mask (weight_orig × weight_mask).
    Used to avoid double-counting and to read through masks correctly.
    """
    stats = {}
    for module in model.modules():
        if hasattr(module, 'weight_mask') and hasattr(module, 'weight_orig'):
            effective = module.weight_orig * module.weight_mask
            stats[id(module.weight_orig)] = (
                effective.numel(),
                int((effective != 0).sum().item()),
            )
    return stats


def count_parameters(model):
    """
    Returns (total_params, nonzero_params).

    When pruning masks are still active, reads weight_orig × weight_mask so
    that the count reflects true effective sparsity during fine-tuning.
    After remove_pruning_masks() the baked-in zeros are counted directly.
    """
    masked = _masked_stats(model)
    total = nonzero = 0

    for param in model.parameters():
        pid = id(param)
        if pid in masked:
            n, nz = masked[pid]
        else:
            n  = param.numel()
            try:
                nz = int((param != 0).sum().item())
            except Exception:
                nz = n
        total   += n
        nonzero += nz

    return total, nonzero


def get_model_size_mb(model):
    import io
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return len(buffer.getvalue()) / (1024 ** 2)


# =============================================================================
# 1. PRUNING
# =============================================================================
# apply_pruning() attaches masks but does NOT remove them.
# This allows fine-tuning with sparsity enforced (forward pass uses
# weight = weight_orig × weight_mask).
#
# CALLER MUST call remove_pruning_masks(model) AFTER fine-tuning so that
# zeros are baked in before the final evaluation and logging.
#
# RNN weights (weight_ih_l*, weight_hh_l*) are not compatible with
# torch.nn.utils.prune mask hooks so they receive a manual threshold mask.
# =============================================================================

def remove_pruning_masks(model):
    """
    Bakes pruning masks permanently into weight tensors (weight = weight_orig × mask).
    Call AFTER fine-tuning is complete, before final evaluation.
    """
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass    # not pruned — skip
    return model


def _prune_standard_layers(model, compression_rate):
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
            if module.weight is not None:
                prune.l1_unstructured(
                    module, name='weight', amount=compression_rate
                )


def _prune_rnn_layers(model, compression_rate):
    """Threshold-based mask applied in-place to RNN packed weight tensors."""
    for name, param in model.named_parameters():
        if 'weight_ih' in name or 'weight_hh' in name:
            with torch.no_grad():
                w   = param.data
                thr = torch.quantile(w.abs(), compression_rate)
                param.data = w * (w.abs() > thr)


def apply_pruning(model, compression_rate):
    """
    Attaches L1-unstructured pruning masks to all Linear / Conv1d / Conv2d
    layers and applies in-place magnitude masking to RNN weight tensors.

    IMPORTANT: masks are intentionally kept live so fine-tuning enforces
    sparsity.  Call remove_pruning_masks() AFTER fine-tuning.

    Args
        model            : any of the 9 models
        compression_rate : fraction of weights to zero (0.15 → 15 %)

    Returns
        pruned model (deep copy; original unchanged)
    """
    model = copy.deepcopy(model)
    _prune_standard_layers(model, compression_rate)   # attaches masks
    _prune_rnn_layers(model, compression_rate)         # in-place on RNNs
    # masks are NOT removed here — caller removes after fine-tuning
    return model


# =============================================================================
# 2. LOW-RANK FACTORIZATION  —  truncated SVD
# =============================================================================
# compression_rate = fraction of singular values to RETAIN:
#   0.15 → keep top 15 %  (most aggressive)
#   0.75 → keep top 75 %  (least aggressive)
#
# Branch A  nn.Linear  → two smaller nn.Linear layers
# Branch B  nn.Conv2d  → (kH×kW conv) + (1×1 conv)
# Branch B2 nn.Conv1d  → (kW conv)    + (1×1 conv)   [M3]
# Branch C  RNN        → in-place low-rank weight approximation
#
# SKIP CONDITION (v3 fix):
#   Conv2d: old = min(in_channels, out_channels)   ← misses TextCNN Conv2d(1,100,…)
#           new = min(out_c, in_c × kH × kW)       ← reflects 2-D SVD matrix
#   Conv1d: min(out_c, in_c × kW)
# =============================================================================

def _factorize_linear(layer, rank_ratio):
    W    = layer.weight.data.float()
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    rank = max(1, int(rank_ratio * S.size(0)))

    l1 = nn.Linear(layer.in_features, rank, bias=False)
    l1.weight.data = Vh[:rank, :]

    l2 = nn.Linear(rank, layer.out_features, bias=(layer.bias is not None))
    l2.weight.data = U[:, :rank] * S[:rank].unsqueeze(0)
    if layer.bias is not None:
        l2.bias.data = layer.bias.data.clone()

    return nn.Sequential(l1, l2)


def _factorize_conv2d(layer, rank_ratio):
    W = layer.weight.data.float()
    out_c, in_c, kH, kW = W.shape
    U, S, Vh = torch.linalg.svd(W.view(out_c, -1), full_matrices=False)
    rank = max(1, int(rank_ratio * S.size(0)))

    c1 = nn.Conv2d(in_c, rank, kernel_size=(kH, kW),
                   stride=layer.stride, padding=layer.padding,
                   dilation=layer.dilation, groups=layer.groups, bias=False)
    c1.weight.data = Vh[:rank, :].view(rank, in_c, kH, kW)

    c2 = nn.Conv2d(rank, out_c, kernel_size=1, bias=(layer.bias is not None))
    c2.weight.data = (U[:, :rank] * S[:rank].unsqueeze(0)).view(out_c, rank, 1, 1)
    if layer.bias is not None:
        c2.bias.data = layer.bias.data.clone()

    return nn.Sequential(c1, c2)


def _factorize_conv1d(layer, rank_ratio):
    W = layer.weight.data.float()
    out_c, in_c, kW = W.shape
    U, S, Vh = torch.linalg.svd(W.view(out_c, -1), full_matrices=False)
    rank = max(1, int(rank_ratio * S.size(0)))

    c1 = nn.Conv1d(in_c, rank, kernel_size=kW,
                   stride=layer.stride, padding=layer.padding,
                   dilation=layer.dilation, groups=layer.groups, bias=False)
    c1.weight.data = Vh[:rank, :].view(rank, in_c, kW)

    c2 = nn.Conv1d(rank, out_c, kernel_size=1, bias=(layer.bias is not None))
    c2.weight.data = (U[:, :rank] * S[:rank].unsqueeze(0)).view(out_c, rank, 1)
    if layer.bias is not None:
        c2.bias.data = layer.bias.data.clone()

    return nn.Sequential(c1, c2)


def _factorize_rnn_weights(model, rank_ratio):
    """Branch C: in-place low-rank approximation of RNN packed weight tensors."""
    for name, param in model.named_parameters():
        if 'weight_ih' in name or 'weight_hh' in name:
            with torch.no_grad():
                W = param.data.float()
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                rank = max(1, int(rank_ratio * S.size(0)))
                param.data = (
                    (U[:, :rank] * S[:rank].unsqueeze(0)) @ Vh[:rank, :]
                ).to(param.dtype)


def _replace_layers_recursive(parent, rank_ratio):
    """
    Depth-first traversal replacing Linear / Conv2d / Conv1d with factorized pairs.

    Skip condition uses 2-D SVD matrix dimensions so that Conv2d layers with
    small in_channels (e.g. TextCNN Conv2d(1,100,k,128)) are not incorrectly
    skipped.
    """
    for name, module in list(parent.named_children()):
        if isinstance(module, nn.Linear):
            if min(module.in_features, module.out_features) > 16:
                setattr(parent, name, _factorize_linear(module, rank_ratio))
            # small Linear (e.g. 300×2 classifier) → skip, no recursion needed

        elif isinstance(module, nn.Conv2d):
            out_c   = module.weight.shape[0]
            flat_in = module.in_channels * module.weight.shape[2] * module.weight.shape[3]
            if min(out_c, flat_in) > 16:
                setattr(parent, name, _factorize_conv2d(module, rank_ratio))
            # else skip — factorizing would increase params

        elif isinstance(module, nn.Conv1d):
            out_c   = module.weight.shape[0]
            flat_in = module.in_channels * module.weight.shape[2]
            if min(out_c, flat_in) > 16:
                setattr(parent, name, _factorize_conv1d(module, rank_ratio))

        else:
            # Recurse into containers (Sequential, ModuleList, attention blocks…)
            _replace_layers_recursive(module, rank_ratio)


def apply_low_rank_factorization(model, compression_rate):
    """
    Replaces Linear / Conv2d / Conv1d with truncated-SVD layer pairs, and
    approximates RNN weight matrices in-place.

    Args
        model            : any of the 9 models
        compression_rate : fraction of singular values to RETAIN
                           0.15 = most aggressive, 0.75 = least aggressive

    Returns
        factorized model (deep copy; original unchanged)
    """
    model = copy.deepcopy(model)
    rank_ratio = max(0.05, 1.0 - compression_rate)
    _replace_layers_recursive(model, rank_ratio)   # branches A, B, B2
    _factorize_rnn_weights(model, rank_ratio)      # branch C
    return model


# =============================================================================
# Main Compression Method
# =============================================================================

def compress_model(model, technique, compression_rate):
    """
    Apply one compression technique to a model.

    Returns
        compressed model (deep copy)

    Note: 'pruning' returns a model with live masks — call
          remove_pruning_masks() AFTER fine-tuning.
    """
    t = technique.lower().strip()
    if t == 'pruning':
        return apply_pruning(model, compression_rate)
    if t in ('factorization', 'low_rank', 'low-rank'):
        return apply_low_rank_factorization(model, compression_rate)
    raise ValueError(
        f"Unknown technique '{technique}'. "
        "Choose from: 'pruning' or 'factorization'."
    )

In [ ]:
import os, sys, time, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
import torchaudio
import torchaudio.transforms as AT
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

# ── Model files must be in the same directory as this notebook ──
sys.path.insert(0, '.')
from audio_models      import M3, build_ast, AudioLSTM
from sentiment_models  import TextCNN, build_bert, BiLSTM
from translation_models import GRUSeq2Seq, TranslationTransformer, ConvS2S
print("All imports OK")

In [ ]:
DEVICE             = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COMPRESSION_RATES  = [0.15, 0.30, 0.45, 0.60, 0.75]
TECHNIQUES         = ['pruning', 'quantization', 'factorization']
SEED               = 42
SPLIT_RATIO        = 0.70          # 70 % train, 30 % test
FINETUNE_RATIO     = 0.30          # fine-tune compressed models for 30 % of original epochs
DATA_DIR           = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device : {DEVICE}")
print(f"Rates  : {COMPRESSION_RATES}")

In [ ]:
results_log = []
RESULTS_CSV = Path('./results_compression_experiments.csv')
criterion_ce  = nn.CrossEntropyLoss()
def log_result(task, model_name, technique, rate,
               total_params, nonzero_params, size_mb,
               train_time_s, test_time_s, metric, baseline_metric=None):
    acc_loss = None
    if baseline_metric is not None:
        acc_loss = round(baseline_metric - metric, 4)
    row = {
        'task'             : task,
        'model'            : model_name,
        'technique'        : technique,
        'compression_rate' : rate,
        'total_params'     : total_params,
        'nonzero_params'   : nonzero_params,
        'size_mb'          : round(size_mb, 3),
        'train_time_s'     : round(train_time_s, 2),
        'test_time_s'      : round(test_time_s, 4),
        'metric'           : round(metric, 4),
        'accuracy_loss'    : acc_loss,
    }
    results_log.append(row)
    # Write header only on the first row, then append subsequent rows
    write_header = not RESULTS_CSV.exists()
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)
    tag = f"[{technique} @ {int(rate*100)}%]" if technique != 'original' else '[original]'
    print(f"  {model_name:30s} {tag:22s}  metric={metric:.4f}"
          + (f"  loss={acc_loss:+.4f}" if acc_loss is not None else ""))
    print(f"  └─ saved to {RESULTS_CSV}  (total rows: {len(results_log)})")

def results_df():
    # Re-read from CSV so the df reflects what was actually persisted
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame(results_log)

In [ ]:
class NoamScheduler:
    """Vaswani et al. 2017 learning-rate schedule for the Transformer."""
    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self._step        = 0

    def step(self):
        self._step += 1
        lr = self.d_model ** -0.5 * min(
            self._step ** -0.5,
            self._step * self.warmup_steps ** -1.5
        )
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    # Proxy so training loop can call scheduler.step() uniformly
    def __call__(self): self.step()

In [ ]:
# ── Classification (Audio + Sentiment) ──────────────────────────────────────

def train_epoch_cls(model, loader, optimizer, criterion, device, grad_clip=None,
                    scheduler=None, is_hf=False):
    model.train()
    total_loss = 0.0
    for batch in loader:
        inputs, labels = batch
        optimizer.zero_grad()
        if is_hf:
            # HuggingFace model: inputs is a dict of tensors
            outputs = model(**{k: v.to(device) for k, v in inputs.items()})
            logits  = outputs.logits
        elif isinstance(inputs, (list, tuple)):
            logits = model(*[x.to(device) for x in inputs])
        else:
            logits = model(inputs.to(device))
        loss = criterion(logits, labels.to(device))
        loss.backward()
        if grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        if scheduler is not None and not isinstance(scheduler, NoamScheduler):
            # NoamScheduler is stepped inside the translation loop
            pass
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_cls(model, loader, criterion, device, is_hf=False):
    model.eval()
    correct = total = 0
    total_loss = 0.0
    for batch in loader:
        inputs, labels = batch
        if is_hf:
            outputs = model(**{k: v.to(device) for k, v in inputs.items()})
            logits  = outputs.logits
        elif isinstance(inputs, (list, tuple)):
            logits = model(*[x.to(device) for x in inputs])
        else:
            logits = model(inputs.to(device))
        loss = criterion(logits, labels.to(device))
        total_loss += loss.item()
        preds    = logits.argmax(1)
        correct += (preds == labels.to(device)).sum().item()
        total   += labels.size(0)
    return correct / total, total_loss / len(loader)


def train_cls(model_name, task, model, train_loader, test_loader,
              optimizer, criterion, n_epochs, device,
              grad_clip=None, scheduler=None, is_hf=False,
              sched_on_plateau=False):
    """Full training run; returns (model, train_time_s, test_acc, test_time_s)."""
    t0 = time.time()
    best_acc = 0.0
    for epoch in range(1, n_epochs + 1):
        tr_loss = train_epoch_cls(model, train_loader, optimizer, criterion,
                                  device, grad_clip, scheduler, is_hf)
        val_acc, val_loss = eval_cls(model, test_loader, criterion, device, is_hf)
        if sched_on_plateau and scheduler is not None:
            scheduler.step(val_loss)
        elif scheduler is not None and not isinstance(scheduler, NoamScheduler):
            scheduler.step()
        if val_acc > best_acc:
            best_acc = val_acc
        if epoch % max(1, n_epochs // 5) == 0:
            print(f"    epoch {epoch:3d}/{n_epochs}  loss={tr_loss:.4f}  acc={val_acc:.4f}")
    train_time = time.time() - t0
    t1 = time.time()
    final_acc, _ = eval_cls(model, test_loader, criterion, device, is_hf)
    test_time = time.time() - t1
    return model, train_time, final_acc, test_time


# ── Translation ──────────────────────────────────────────────────────────────

def train_epoch_mt(model, loader, optimizer, criterion, device,
                   grad_clip=None, noam=None, tf_ratio=0.5):
    model.train()
    total_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        if isinstance(model, GRUSeq2Seq):
            # GRUSeq2Seq forward takes full tgt (with SOS); output[:,0] is zeros
            out = model(src, tgt, teacher_forcing_ratio=tf_ratio)
            out = out[:, 1:, :]   # skip position 0 (zeros)
        else:
            out = model(src, tgt_in)
        loss = criterion(out.reshape(-1, out.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        if grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        if noam:
            noam.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def greedy_decode(model, src, max_len, sos_idx, eos_idx, pad_idx, device):
    """Greedy decoding for all three translation models."""
    src = src.to(device)
    bs  = src.size(0)
    if isinstance(model, GRUSeq2Seq):
        enc_out, hidden = model.encoder(src)
        token = torch.full((bs,), sos_idx, dtype=torch.long, device=device)
        seq   = []
        for _ in range(max_len):
            pred, hidden = model.decoder(token, hidden, enc_out)
            token = pred.argmax(1)
            seq.append(token)
        return torch.stack(seq, dim=1)   # (bs, max_len)
    else:
        tgt = torch.full((bs, 1), sos_idx, dtype=torch.long, device=device)
        for _ in range(max_len):
            out   = model(src, tgt)
            nxt   = out[:, -1, :].argmax(1, keepdim=True)
            tgt   = torch.cat([tgt, nxt], dim=1)
            if (nxt.squeeze(1) == eos_idx).all():
                break
        return tgt[:, 1:]   # strip SOS


@torch.no_grad()
def eval_mt(model, loader, criterion, tgt_vocab, inv_tgt_vocab,
            sos_idx, eos_idx, pad_idx, device, max_decode=50):
    """Returns (bleu_score_0_to_1, avg_loss, eval_time_s)."""
    from sacrebleu.metrics import BLEU as SacreBLEU
    model.eval()
    hyps = []
    refs = []
    total_loss = 0.0
    t0 = time.time()
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        # loss
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        if isinstance(model, GRUSeq2Seq):
            out = model(src, tgt, teacher_forcing_ratio=0.0)
            out = out[:, 1:, :]
        else:
            out = model(src, tgt_in)
        loss = criterion(out.reshape(-1, out.size(-1)), tgt_out.reshape(-1))
        total_loss += loss.item()
        # decode
        pred_ids = greedy_decode(model, src, max_decode, sos_idx, eos_idx, pad_idx, device)
        ref_ids  = tgt[:, 1:]   # strip SOS
        for p_row, r_row in zip(pred_ids.cpu().tolist(), ref_ids.cpu().tolist()):
            def ids_to_str(ids):
                tokens = []
                for i in ids:
                    if i == eos_idx or i == pad_idx: break
                    tokens.append(inv_tgt_vocab.get(i, '<unk>'))
                return ' '.join(tokens)
            hyps.append(ids_to_str(p_row))
            refs.append(ids_to_str(r_row))
    eval_time = time.time() - t0
    bleu = SacreBLEU(effective_order=True)
    score = bleu.corpus_score(hyps, [refs]).score / 100.0   # normalise 0-1
    return score, total_loss / len(loader), eval_time


def train_mt(model_name, model, train_loader, test_loader,
             optimizer, criterion, n_epochs, device,
             tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx,
             grad_clip=None, noam=None, sched=None, tf_ratio=0.5,
             sched_on_plateau=False):
    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        tr_loss = train_epoch_mt(model, train_loader, optimizer, criterion,
                                  device, grad_clip, noam, tf_ratio)
        if sched_on_plateau and sched:
            val_bleu, val_loss, _ = eval_mt(model, test_loader, criterion,
                                             tgt_vocab, inv_tgt_vocab,
                                             sos_idx, eos_idx, pad_idx, device)
            sched.step(val_loss)
        if epoch % max(1, n_epochs // 4) == 0:
            print(f"    epoch {epoch:3d}/{n_epochs}  loss={tr_loss:.4f}")
    train_time = time.time() - t0
    bleu, _, test_time = eval_mt(model, test_loader, criterion,
                                  tgt_vocab, inv_tgt_vocab,
                                  sos_idx, eos_idx, pad_idx, device)
    return model, train_time, bleu, test_time

In [ ]:
def run_compression_cls(model_name, task, trained_model, train_loader, test_loader,
                        make_optimizer, make_scheduler, criterion, n_epochs,
                        device, is_hf=False, grad_clip=None, sched_on_plateau=False):
    """Applies the 2 techniques x 5 rates to a trained classification model."""
    t0 = time.time()
    base_acc, _ = eval_cls(trained_model, test_loader, criterion, device, is_hf)
    base_test_t = time.time() - t0
    tp, nz = count_parameters(trained_model)
    log_result(task, model_name, 'original', 0.0,
               tp, nz, get_model_size_mb(trained_model),
               0.0, base_test_t, base_acc)

    ft_epochs = max(1, int(n_epochs * FINETUNE_RATIO))

    for technique in TECHNIQUES:
        for rate in COMPRESSION_RATES:
            compressed  = compress_model(trained_model, technique, rate)
            eval_device = device
            compressed  = compressed.to(eval_device)

            if technique == 'pruning':
                opt   = make_optimizer(compressed)
                sched = make_scheduler(opt) if make_scheduler else None
                _, train_t, _, _ = train_cls(
                    model_name, task, compressed, train_loader, test_loader,
                    opt, criterion, ft_epochs, eval_device,
                    grad_clip, sched, is_hf, sched_on_plateau
                )
                # Bake zeros in AFTER fine-tuning so count_parameters sees them
                remove_pruning_masks(compressed)

            else:  # factorization
                opt   = make_optimizer(compressed)
                sched = make_scheduler(opt) if make_scheduler else None
                _, train_t, _, _ = train_cls(
                    model_name, task, compressed, train_loader, test_loader,
                    opt, criterion, ft_epochs, eval_device,
                    grad_clip, sched, is_hf, sched_on_plateau
                )

            t1 = time.time()
            acc, _ = eval_cls(compressed, test_loader, criterion, eval_device, is_hf)
            test_t = time.time() - t1
            tp, nz = count_parameters(compressed)
            log_result(task, model_name, technique, rate,
                       tp, nz, get_model_size_mb(compressed),
                       train_t, test_t, acc, base_acc)
            del compressed
            if torch.cuda.is_available(): torch.cuda.empty_cache()


def run_compression_mt(model_name, trained_model, train_loader, test_loader,
                       make_optimizer, make_scheduler, criterion, n_epochs,
                       device, tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx,
                       grad_clip=None, tf_ratio=0.5, sched_on_plateau=False):
    """Applies the 2 techniques x 5 rates to a trained translation model."""
    t0 = time.time()
    base_bleu, _, base_test_t = eval_mt(
        trained_model, test_loader, criterion,
        tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx, device
    )
    tp, nz = count_parameters(trained_model)
    log_result('translation', model_name, 'original', 0.0,
               tp, nz, get_model_size_mb(trained_model),
               0.0, base_test_t, base_bleu)

    ft_epochs = max(1, int(n_epochs * FINETUNE_RATIO))
    if model_name == 'TranslationTransformer':
        base_model = copy.deepcopy(trained_model)

    for technique in TECHNIQUES:
        for rate in COMPRESSION_RATES:
            if model_name == 'TranslationTransformer':
                compressed  = compress_model(base_model, technique, rate)
            else:
                compressed  = compress_model(trained_model, technique, rate)
            eval_device = device
            compressed  = compressed.to(eval_device)

            if technique == 'pruning':
                opt  = make_optimizer(compressed)
                noam = make_scheduler(opt) if make_scheduler else None
                _, train_t, _, _ = train_mt(
                    model_name, compressed, train_loader, test_loader,
                    opt, criterion, ft_epochs, eval_device,
                    tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx,
                    grad_clip,
                    noam if not sched_on_plateau else None,
                    noam if sched_on_plateau else None,
                    tf_ratio, sched_on_plateau
                )
                remove_pruning_masks(compressed)   # AFTER fine-tuning

            else:  # factorization
                opt  = make_optimizer(compressed)
                noam = make_scheduler(opt) if make_scheduler else None
                _, train_t, _, _ = train_mt(
                    model_name, compressed, train_loader, test_loader,
                    opt, criterion, ft_epochs, eval_device,
                    tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx,
                    grad_clip,
                    noam if not sched_on_plateau else None,
                    noam if sched_on_plateau else None,
                    tf_ratio, sched_on_plateau
                )

            bleu, _, test_t = eval_mt(
                compressed, test_loader, criterion,
                tgt_vocab, inv_tgt_vocab, sos_idx, eos_idx, pad_idx, eval_device
            )
            tp, nz = count_parameters(compressed)
            log_result('translation', model_name, technique, rate,
                       tp, nz, get_model_size_mb(compressed),
                       train_t, test_t, bleu, base_bleu)
            del compressed
            if torch.cuda.is_available(): torch.cuda.empty_cache()

## Task 1 — Audio Classification
**Dataset:** Google Speech Commands (35 classes)  
**Split:** 70 / 30

In [ ]:
SAMPLE_RATE = 16000
N_MELS      = 64
FIXED_LEN   = 16000   # 1 second at 16 kHz

SC_CLASSES = sorted([
    'backward','bed','bird','cat','dog','down','eight','five','follow','forward',
    'four','go','happy','house','learn','left','marvin','nine','no','off','on',
    'one','right','seven','sheila','six','stop','three','tree','two','up',
    'visual','wow','yes','zero'
])
SC_LABEL2IDX = {c: i for i, c in enumerate(SC_CLASSES)}
NUM_AUDIO_CLASSES = len(SC_CLASSES)   # 35

mel_transform = AT.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_mels=N_MELS,
    n_fft=400, hop_length=160
)


class SpeechCommandsDataset(Dataset):
    """Wraps torchaudio SPEECHCOMMANDS; returns (waveform, mel, label)."""
    def __init__(self, root, subset=None, download=True):
        self.ds = torchaudio.datasets.SPEECHCOMMANDS(
            root=str(root), download=download, subset=subset
        )

    def __len__(self):  return len(self.ds)

    def __getitem__(self, idx):
        waveform, sr, label, *_ = self.ds[idx]
        if sr != SAMPLE_RATE:
            waveform = AT.Resample(sr, SAMPLE_RATE)(waveform)
        # Pad / crop to FIXED_LEN
        n = waveform.size(1)
        if n < FIXED_LEN:
            waveform = torch.nn.functional.pad(waveform, (0, FIXED_LEN - n))
        else:
            waveform = waveform[:, :FIXED_LEN]
        # Mel-spectrogram
        mel = mel_transform(waveform)
        log_mel = (mel.clamp(min=1e-9)).log2().squeeze(0).T   # (time, n_mels)
        label_idx = SC_LABEL2IDX.get(label, 0)
        return waveform, log_mel, label_idx


def audio_collate(batch):
    waveforms, mels, labels = zip(*batch)
    return (torch.stack(waveforms),          # (B, 1, FIXED_LEN)
            torch.stack(mels),                # (B, time, n_mels)
            torch.tensor(labels, dtype=torch.long))


print("Loading Speech Commands …")
sc_train = SpeechCommandsDataset(DATA_DIR, subset='training')
sc_val   = SpeechCommandsDataset(DATA_DIR, subset='validation')
sc_test  = SpeechCommandsDataset(DATA_DIR, subset='testing')
sc_full  = ConcatDataset([sc_train, sc_val, sc_test])
n_sc_train = int(SPLIT_RATIO * len(sc_full))
n_sc_test  = len(sc_full) - n_sc_train
sc_train_set, sc_test_set = random_split(
    sc_full, [n_sc_train, n_sc_test],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"  total={len(sc_full):,}  train={n_sc_train:,}  test={n_sc_test:,}")

### 1b — M3 (CNN)

In [ ]:
AUDIO_BATCH = 256

def make_m3_loaders():
    # M3 uses raw waveform: collate returns (waveform, mel, label) -> we use waveform
    class WaveOnly(Dataset):
        def __init__(self, ds): self.ds = ds
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            w, _, lbl = self.ds[i]; return w, lbl

    tr = DataLoader(WaveOnly(sc_train_set), batch_size=AUDIO_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
    te = DataLoader(WaveOnly(sc_test_set),  batch_size=AUDIO_BATCH, shuffle=False, num_workers=2, pin_memory=True)
    return tr, te

m3_tr, m3_te = make_m3_loaders()

# Hyperparameters — Dai et al. 2017
M3_EPOCHS = 5
m3_model  = M3(num_classes=NUM_AUDIO_CLASSES).to(DEVICE)

def m3_opt(m):
    return optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-4)
def m3_sched(opt):
    return optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.1)

print("Training M3 …")
m3_opt_inst   = m3_opt(m3_model)
m3_sched_inst = m3_sched(m3_opt_inst)
criterion_ce  = nn.CrossEntropyLoss()

m3_model, m3_tr_t, m3_acc, m3_te_t = train_cls(
    'M3', 'audio', m3_model, m3_tr, m3_te,
    m3_opt_inst, criterion_ce, M3_EPOCHS, DEVICE,
    scheduler=m3_sched_inst
)
tp, nz = count_parameters(m3_model)
log_result('audio','M3','original',0.0, tp, nz, get_model_size_mb(m3_model), m3_tr_t, m3_te_t, m3_acc)
print(f"M3 baseline  acc={m3_acc:.4f}  train={m3_tr_t:.1f}s")

In [ ]:
print("Compressing M3 …")
run_compression_cls('M3', 'audio', m3_model, m3_tr, m3_te,
                    m3_opt, m3_sched, criterion_ce, M3_EPOCHS, DEVICE)

### 1c — AST (Transformer)

In [ ]:
from transformers import ASTFeatureExtractor

ast_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

class ASTDataset(Dataset):
    """Wraps Speech Commands split; returns HF feature dict + label."""
    def __init__(self, split_ds):
        self.ds = split_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        w, _, lbl = self.ds[i]
        feat = ast_extractor(
            w.squeeze(0).numpy(), sampling_rate=SAMPLE_RATE, return_tensors='pt'
        )
        # feat['input_values']: (1, n_mels, time_frames)
        return {k: v.squeeze(0) for k, v in feat.items()}, lbl

def ast_collate(batch):
    feats, labels = zip(*batch)
    batched = {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
    return batched, torch.tensor(labels, dtype=torch.long)

AST_BATCH  = 32
ast_tr_ds  = ASTDataset(sc_train_set)
ast_te_ds  = ASTDataset(sc_test_set)
ast_tr     = DataLoader(ast_tr_ds, batch_size=AST_BATCH, shuffle=True,  collate_fn=ast_collate, num_workers=2)
ast_te     = DataLoader(ast_te_ds, batch_size=AST_BATCH, shuffle=False, collate_fn=ast_collate, num_workers=2)

AST_EPOCHS = 5
ast_model  = build_ast(num_classes=NUM_AUDIO_CLASSES, pretrained=True).to(DEVICE)

def ast_opt(m):
    return optim.Adam(m.parameters(), lr=1e-5, weight_decay=5e-7)
def ast_sched(opt):
    return optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

print("Training AST …")
ast_opt_i   = ast_opt(ast_model)
ast_sched_i = ast_sched(ast_opt_i)

ast_model, ast_tr_t, ast_acc, ast_te_t = train_cls(
    'AST', 'audio', ast_model, ast_tr, ast_te,
    ast_opt_i, criterion_ce, AST_EPOCHS, DEVICE,
    scheduler=ast_sched_i, is_hf=True, sched_on_plateau=True
)
tp, nz = count_parameters(ast_model)
log_result('audio','AST','original',0.0, tp, nz, get_model_size_mb(ast_model), ast_tr_t, ast_te_t, ast_acc)
print(f"AST baseline  acc={ast_acc:.4f}  train={ast_tr_t:.1f}s")

In [ ]:
print("Compressing AST …")
run_compression_cls('AST', 'audio', ast_model, ast_tr, ast_te,
                    ast_opt, ast_sched, criterion_ce, AST_EPOCHS, DEVICE,
                    is_hf=True, sched_on_plateau=True)

### 1d — AudioLSTM (RNN)

In [ ]:
class MelDataset(Dataset):
    def __init__(self, split_ds):
        self.ds = split_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        _, mel, lbl = self.ds[i]; return mel, lbl

def mel_collate(batch):
    mels, labels = zip(*batch)
    mels_padded = nn.utils.rnn.pad_sequence(mels, batch_first=True)
    return mels_padded, torch.tensor(labels, dtype=torch.long)

LSTM_AUDIO_BATCH = 128
lstm_audio_tr = DataLoader(MelDataset(sc_train_set), batch_size=LSTM_AUDIO_BATCH,
                           shuffle=True,  collate_fn=mel_collate, num_workers=2)
lstm_audio_te = DataLoader(MelDataset(sc_test_set),  batch_size=LSTM_AUDIO_BATCH,
                           shuffle=False, collate_fn=mel_collate, num_workers=2)

ALSTM_EPOCHS = 5
alstm_model  = AudioLSTM(input_size=N_MELS, hidden_size=128,
                          num_layers=2, num_classes=NUM_AUDIO_CLASSES).to(DEVICE)

def alstm_opt(m):
    return optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
def alstm_sched(opt):
    return optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

print("Training AudioLSTM …")
alstm_o = alstm_opt(alstm_model)
alstm_s = alstm_sched(alstm_o)

alstm_model, alstm_tr_t, alstm_acc, alstm_te_t = train_cls(
    'AudioLSTM', 'audio', alstm_model, lstm_audio_tr, lstm_audio_te,
    alstm_o, criterion_ce, ALSTM_EPOCHS, DEVICE,
    grad_clip=1.0, scheduler=alstm_s
)
tp, nz = count_parameters(alstm_model)
log_result('audio','AudioLSTM','original',0.0, tp, nz, get_model_size_mb(alstm_model), alstm_tr_t, alstm_te_t, alstm_acc)
print(f"AudioLSTM baseline  acc={alstm_acc:.4f}  train={alstm_tr_t:.1f}s")

In [ ]:
print("Compressing AudioLSTM …")
run_compression_cls('AudioLSTM','audio', alstm_model, lstm_audio_tr, lstm_audio_te,
                    alstm_opt, alstm_sched, criterion_ce, ALSTM_EPOCHS, DEVICE, grad_clip=1.0)

## Task 2 — Sentiment Analysis
**Dataset:** IMDB (binary: positive / negative)  
**Split:** 70 / 30

In [ ]:
from datasets import load_dataset, concatenate_datasets
from collections import Counter

IMDB_VOCAB_SIZE = 25000
IMDB_MAX_LEN    = 400
PAD_IDX         = 0
UNK_IDX         = 1

print("Loading IMDB …")
imdb_raw  = load_dataset("imdb")
# Concatenate train (25k) + test (25k) = 50k total, then 70/30 split
imdb_full = concatenate_datasets([imdb_raw['train'], imdb_raw['test']])
imdb_full = imdb_full.shuffle(seed=SEED)
n_tr      = int(SPLIT_RATIO * len(imdb_full))
im_train  = imdb_full.select(range(n_tr))
im_test   = imdb_full.select(range(n_tr, len(imdb_full)))
print(f"  total={len(imdb_full):,}  train={len(im_train):,}  test={len(im_test):,}")

# Build vocabulary from training portion only (avoids data leakage)
def tokenize(text): return text.lower().split()

counter = Counter()
for item in im_train:
    counter.update(tokenize(item['text']))

vocab = {'<pad>': 0, '<unk>': 1}
for word, _ in counter.most_common(IMDB_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

ACTUAL_VOCAB_SIZE = len(vocab)
print(f"  vocab size = {ACTUAL_VOCAB_SIZE:,}")

def text_to_tensor(text):
    ids = [vocab.get(w, UNK_IDX) for w in tokenize(text)][:IMDB_MAX_LEN]
    t   = torch.zeros(IMDB_MAX_LEN, dtype=torch.long)
    t[:len(ids)] = torch.tensor(ids, dtype=torch.long)
    return t

class IMDBDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        return text_to_tensor(item['text']), int(item['label'])

# imdb_tr / imdb_te are shared by TextCNN and BiLSTM directly
imdb_tr = DataLoader(IMDBDataset(im_train), batch_size=64,
                     shuffle=True,  num_workers=2, pin_memory=True)
imdb_te = DataLoader(IMDBDataset(im_test),  batch_size=64,
                     shuffle=False, num_workers=2, pin_memory=True)
print("IMDB data loaders ready")

### 2b — TextCNN (CNN)

In [ ]:
tcnn_tr = imdb_tr
tcnn_te = imdb_te

TCNN_EPOCHS = 5
tcnn_model  = TextCNN(vocab_size=ACTUAL_VOCAB_SIZE, embed_dim=128,
                       num_classes=2, num_filters=100,
                       filter_sizes=(3,4,5), dropout=0.5).to(DEVICE)

def tcnn_opt(m):  return optim.Adam(m.parameters(), lr=1e-3)
def tcnn_sched(opt): return None

print("Training TextCNN …")
tcnn_o = tcnn_opt(tcnn_model)
tcnn_model, tcnn_tr_t, tcnn_acc, tcnn_te_t = train_cls(
    'TextCNN', 'sentiment', tcnn_model, tcnn_tr, tcnn_te,
    tcnn_o, criterion_ce, TCNN_EPOCHS, DEVICE
)
tp, nz = count_parameters(tcnn_model)
log_result('sentiment','TextCNN','original',0.0, tp, nz, get_model_size_mb(tcnn_model), tcnn_tr_t, tcnn_te_t, tcnn_acc)
print(f"TextCNN baseline  acc={tcnn_acc:.4f}  train={tcnn_tr_t:.1f}s")

In [ ]:
print("Compressing TextCNN …")
run_compression_cls('TextCNN','sentiment', tcnn_model, tcnn_tr, tcnn_te,
                    tcnn_opt, tcnn_sched, criterion_ce, TCNN_EPOCHS, DEVICE)

### 2c — DistilBERT (Transformer)

In [ ]:
from transformers import DistilBertTokenizerFast

dbert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class IMDBHFDataset(Dataset):
    """Returns HuggingFace tokenizer output dict + label.
    Accepts a HuggingFace dataset directly (im_train or im_test).
    """
    def __init__(self, hf_dataset, max_len=512):
        self.ds      = hf_dataset
        self.max_len = max_len
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        enc  = dbert_tokenizer(
            item['text'], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}, int(item['label'])

def hf_collate(batch):
    feats, labels = zip(*batch)
    batched = {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
    return batched, torch.tensor(labels, dtype=torch.long)

DBERT_BATCH = 32
dbert_tr    = DataLoader(IMDBHFDataset(im_train, max_len=512),
                          batch_size=DBERT_BATCH, shuffle=True,  collate_fn=hf_collate, num_workers=2)
dbert_te    = DataLoader(IMDBHFDataset(im_test,  max_len=512),
                          batch_size=DBERT_BATCH, shuffle=False, collate_fn=hf_collate, num_workers=2)

DBERT_EPOCHS  = 5
dbert_model   = build_bert(num_classes=2, pretrained=True).to(DEVICE)
total_steps   = len(dbert_tr) * DBERT_EPOCHS
warmup_steps  = int(0.1 * total_steps)

def dbert_opt(m):
    return optim.AdamW(m.parameters(), lr=2e-5, weight_decay=0.01)
def dbert_sched(opt):
    from transformers import get_linear_schedule_with_warmup
    return get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)

print("Training DistilBERT …")
dbert_o = dbert_opt(dbert_model)
dbert_s = dbert_sched(dbert_o)
dbert_model, dbert_tr_t, dbert_acc, dbert_te_t = train_cls(
    'DistilBERT','sentiment', dbert_model, dbert_tr, dbert_te,
    dbert_o, criterion_ce, DBERT_EPOCHS, DEVICE, is_hf=True, scheduler=dbert_s
)
tp, nz = count_parameters(dbert_model)
log_result('sentiment','DistilBERT','original',0.0, tp, nz, get_model_size_mb(dbert_model), dbert_tr_t, dbert_te_t, dbert_acc)
print(f"DistilBERT baseline  acc={dbert_acc:.4f}  train={dbert_tr_t:.1f}s")

In [ ]:
print("Compressing DistilBERT …")
run_compression_cls('DistilBERT','sentiment', dbert_model, dbert_tr, dbert_te,
                    dbert_opt, dbert_sched, criterion_ce, DBERT_EPOCHS, DEVICE, is_hf=True)

### 2d — BiLSTM (RNN)

In [ ]:
bilstm_tr = imdb_tr
bilstm_te = imdb_te

BILSTM_EPOCHS = 5
bilstm_model  = BiLSTM(vocab_size=ACTUAL_VOCAB_SIZE, embed_dim=128,
                        hidden_size=256, num_layers=2, num_classes=2,
                        dropout=0.3).to(DEVICE)

def bilstm_opt(m): return optim.Adam(m.parameters(), lr=1e-3)
def bilstm_sched(opt):
    return optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

print("Training BiLSTM …")
bl_o = bilstm_opt(bilstm_model)
bl_s = bilstm_sched(bl_o)
bilstm_model, bl_tr_t, bl_acc, bl_te_t = train_cls(
    'BiLSTM','sentiment', bilstm_model, bilstm_tr, bilstm_te,
    bl_o, criterion_ce, BILSTM_EPOCHS, DEVICE,
    grad_clip=5.0, scheduler=bl_s, sched_on_plateau=True
)
tp, nz = count_parameters(bilstm_model)
log_result('sentiment','BiLSTM','original',0.0, tp, nz, get_model_size_mb(bilstm_model), bl_tr_t, bl_te_t, bl_acc)
print(f"BiLSTM baseline  acc={bl_acc:.4f}  train={bl_tr_t:.1f}s")


In [ ]:
print("Compressing BiLSTM …")
run_compression_cls('BiLSTM','sentiment', bilstm_model, bilstm_tr, bilstm_te,
                    bilstm_opt, bilstm_sched, criterion_ce, BILSTM_EPOCHS, DEVICE,
                    grad_clip=5.0, sched_on_plateau=True)

## Task 3 — Machine Translation
**Dataset:** IWSLT2017 de→en  
**Split:** 70 / 30  
**Metric:** BLEU (sacrebleu, normalised 0–1)

In [ ]:
from datasets import load_dataset as hf_load_dataset

MT_VOCAB_SIZE = 8000
MT_MAX_LEN    = 80
MT_BATCH      = 64

# opus_books de-en: Parquet-based, ~51k pairs, same translation dict format
# as IWSLT2017 — replaces iwslt2017 which requires deprecated dataset scripts
print("Loading opus_books de-en …")
opus_raw  = hf_load_dataset("Helsinki-NLP/opus_books", "de-en")

# opus_books only has a 'train' split — pool it, then do 70/30 manually
all_pairs = [(item['translation']['de'], item['translation']['en'])
             for item in opus_raw['train']]
print(f"  total pairs = {len(all_pairs):,}")

rng       = np.random.default_rng(SEED)
all_idx   = np.arange(len(all_pairs))
rng.shuffle(all_idx)
n_mt_tr   = int(SPLIT_RATIO * len(all_pairs))
mt_tr_idx = all_idx[:n_mt_tr].tolist()
mt_te_idx = all_idx[n_mt_tr:].tolist()
print(f"  train={n_mt_tr:,}  test={len(mt_te_idx):,}")

# Build word-level vocabularies from training pairs only
SPECIALS_MT = ['<pad>', '<unk>', '<sos>', '<eos>']
MT_PAD_IDX  = 0
MT_UNK_IDX  = 1
MT_SOS_IDX  = 2
MT_EOS_IDX  = 3

def build_mt_vocab(token_iter, max_tokens):
    counter = Counter()
    for tokens in token_iter: counter.update(tokens)
    vocab = {s: i for i, s in enumerate(SPECIALS_MT)}
    for word, _ in counter.most_common(max_tokens - len(SPECIALS_MT)):
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab

def tok_de(t): return t.lower().split()
def tok_en(t): return t.lower().split()

print("  building source (DE) vocab …")
src_vocab = build_mt_vocab(
    (tok_de(all_pairs[i][0]) for i in mt_tr_idx), MT_VOCAB_SIZE
)
print("  building target (EN) vocab …")
tgt_vocab = build_mt_vocab(
    (tok_en(all_pairs[i][1]) for i in mt_tr_idx), MT_VOCAB_SIZE
)
inv_tgt_vocab  = {v: k for k, v in tgt_vocab.items()}
SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)
print(f"  src vocab={SRC_VOCAB_SIZE:,}  tgt vocab={TGT_VOCAB_SIZE:,}")

def encode_mt(text, vocab, tokenize_fn, max_len):
    ids = [vocab.get(t, MT_UNK_IDX) for t in tokenize_fn(text)][:max_len]
    return [MT_SOS_IDX] + ids + [MT_EOS_IDX]

def pad_sequence_mt(ids, max_len):
    t = torch.full((max_len + 2,), MT_PAD_IDX, dtype=torch.long)
    t[:len(ids)] = torch.tensor(ids, dtype=torch.long)
    return t

MT_SEQ_LEN = MT_MAX_LEN + 2   # +2 for SOS + EOS

class MTDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        de, en = all_pairs[self.indices[i]]
        src = pad_sequence_mt(encode_mt(de, src_vocab, tok_de, MT_MAX_LEN), MT_MAX_LEN)
        tgt = pad_sequence_mt(encode_mt(en, tgt_vocab, tok_en, MT_MAX_LEN), MT_MAX_LEN)
        return src, tgt

mt_tr_ds = MTDataset(mt_tr_idx)
mt_te_ds = MTDataset(mt_te_idx)
mt_tr    = DataLoader(mt_tr_ds, batch_size=MT_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
mt_te    = DataLoader(mt_te_ds, batch_size=MT_BATCH, shuffle=False, num_workers=2, pin_memory=True)
mt_crit  = nn.CrossEntropyLoss(ignore_index=MT_PAD_IDX, label_smoothing=0.1)
print("MT data loaders ready")

### 3b — GRUSeq2Seq (RNN)

In [ ]:
GRU_EPOCHS = 5
gru_model  = GRUSeq2Seq(
    src_vocab_size=SRC_VOCAB_SIZE, tgt_vocab_size=TGT_VOCAB_SIZE,
    embed_dim=256, hidden_size=256, enc_layers=1, dropout=0.3
).to(DEVICE)

def gru_opt(m):
    return optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5)
def gru_sched(opt):
    return optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

print("Training GRUSeq2Seq …")
gru_o = gru_opt(gru_model)
gru_s = gru_sched(gru_o)
gru_model, gru_tr_t, gru_bleu, gru_te_t = train_mt(
    'GRUSeq2Seq', gru_model, mt_tr, mt_te,
    gru_o, mt_crit, GRU_EPOCHS, DEVICE,
    tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
    grad_clip=1.0, sched=gru_s, tf_ratio=0.5, sched_on_plateau=True
)
tp, nz = count_parameters(gru_model)
log_result('translation','GRUSeq2Seq','original',0.0, tp, nz, get_model_size_mb(gru_model), gru_tr_t, gru_te_t, gru_bleu)
print(f"GRUSeq2Seq baseline  BLEU={gru_bleu:.4f}  train={gru_tr_t:.1f}s")

In [ ]:
print("Compressing GRUSeq2Seq …")
run_compression_mt('GRUSeq2Seq', gru_model, mt_tr, mt_te,
                   gru_opt, gru_sched, mt_crit, GRU_EPOCHS, DEVICE,
                   tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
                   grad_clip=1.0, tf_ratio=0.5, sched_on_plateau=True)

### 3c — TranslationTransformer (Transformer)

In [ ]:
TF_MT_EPOCHS = 5
tf_mt_model  = TranslationTransformer(
    src_vocab_size=SRC_VOCAB_SIZE, tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=256, nhead=4, num_encoder_layers=3, num_decoder_layers=3,
    dim_feedforward=512, dropout=0.1
).to(DEVICE)

D_MODEL = 256

def tf_mt_opt(m):
    return optim.Adam(m.parameters(), lr=1.0,    # lr is overridden by Noam
                      betas=(0.9, 0.98), eps=1e-9)
def tf_mt_noam(opt):
    return NoamScheduler(opt, d_model=D_MODEL, warmup_steps=4000)

print("Training TranslationTransformer …")
tf_o    = tf_mt_opt(tf_mt_model)
tf_noam = tf_mt_noam(tf_o)
tf_mt_model, tf_tr_t, tf_bleu, tf_te_t = train_mt(
    'TranslationTransformer', tf_mt_model, mt_tr, mt_te,
    tf_o, mt_crit, TF_MT_EPOCHS, DEVICE,
    tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
    grad_clip=1.0, noam=tf_noam, tf_ratio=0.5
)
tp, nz = count_parameters(tf_mt_model)
log_result('translation','TranslationTransformer','original',0.0,
           tp, nz, get_model_size_mb(tf_mt_model), tf_tr_t, tf_te_t, tf_bleu)
print(f"TranslationTransformer baseline  BLEU={tf_bleu:.4f}  train={tf_tr_t:.1f}s")

In [ ]:
print("Compressing TranslationTransformer …")
run_compression_mt('TranslationTransformer', tf_mt_model, mt_tr, mt_te,
                   tf_mt_opt, tf_mt_noam, mt_crit, TF_MT_EPOCHS, DEVICE,
                   tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
                   grad_clip=1.0, tf_ratio=0.5)

### 3d — ConvS2S (CNN)

In [ ]:
CS2S_EPOCHS = 5
cs2s_model  = ConvS2S(
    src_vocab_size=SRC_VOCAB_SIZE, tgt_vocab_size=TGT_VOCAB_SIZE,
    embed_dim=256, hidden_size=512,
    num_encoder_layers=4, num_decoder_layers=4,
    kernel_size=3, dropout=0.1
).to(DEVICE)

def cs2s_opt(m):
    # Gehring et al. 2017: Nesterov SGD
    return optim.SGD(m.parameters(), lr=0.25, momentum=0.99,
                     weight_decay=1e-4, nesterov=True)
def cs2s_sched(opt):
    return optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

print("Training ConvS2S …")
cs_o = cs2s_opt(cs2s_model)
cs_s = cs2s_sched(cs_o)
cs2s_model, cs_tr_t, cs_bleu, cs_te_t = train_mt(
    'ConvS2S', cs2s_model, mt_tr, mt_te,
    cs_o, mt_crit, CS2S_EPOCHS, DEVICE,
    tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
    grad_clip=0.1, sched=cs_s, tf_ratio=0.5, sched_on_plateau=True
)
tp, nz = count_parameters(cs2s_model)
log_result('translation','ConvS2S','original',0.0,
           tp, nz, get_model_size_mb(cs2s_model), cs_tr_t, cs_te_t, cs_bleu)
print(f"ConvS2S baseline  BLEU={cs_bleu:.4f}  train={cs_tr_t:.1f}s")

In [ ]:
print("Compressing ConvS2S …")
run_compression_mt('ConvS2S', cs2s_model, mt_tr, mt_te,
                   cs2s_opt, cs2s_sched, mt_crit, CS2S_EPOCHS, DEVICE,
                   tgt_vocab, inv_tgt_vocab, MT_SOS_IDX, MT_EOS_IDX, MT_PAD_IDX,
                   grad_clip=0.1, tf_ratio=0.5, sched_on_plateau=True)

## Results

In [ ]:
df = results_df()
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)
print(df.to_string(index=False))

In [ ]:
# ── Per-task summaries ────────────────────────────────────────────────────────
for task in ['audio','sentiment','translation']:
    print(f"\n{'='*70}")
    print(f"  {task.upper()}")
    print('='*70)
    sub = df[df['task']==task]
    print(sub[['model','technique','compression_rate','total_params',
               'size_mb','train_time_s','test_time_s','metric','accuracy_loss'
               ]].to_string(index=False))

In [ ]:
# ── Save to CSV ───────────────────────────────────────────────────────────────
out_path = Path('./results_compression_experiments.csv')
df.to_csv(out_path, index=False)
print(f"Results saved to {out_path.resolve()}")
df